In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GOOGLE_API_KEY")

In [ ]:
!pip install -q -U langchain langgraph langchain-google-genai python-dotenv

# 1. Importing Libraries

In [ ]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

# 2.LLM

In [ ]:
llm= ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=secret_value_0
)

# 3.Defining State

In [ ]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str
    rate:float

# 4.Defining Node

## 4.1 Create Content

In [ ]:
def create_outline(state:BlogState)->BlogState:
    title=state["title"]
    prompt=f"Generate a detailed outline on the topic {title}"
    outline=llm.invoke(prompt).content
    state["outline"]=outline
    return state
    

## 4.2 Create Blog

In [ ]:
def create_blog(state:BlogState)->BlogState:
    title=state["title"]
    outline=state["outline"]
    prompt=f"Genetate a blog on the topic {title} on the following \n {outline}"
    content =llm.invoke(prompt).content
    state["content"]=content
    return state

## 4.3 Rate the Blog

In [ ]:
def rate_blog(state:BlogState)->BlogState:
    title=state["title"]
    outline=state["outline"]
    content=state["content"]
    prompt =f"Give the rating out of 10 of the genereted blog {content} on the title {title} from the following content \n {content}"
    rate=llm.invoke(prompt).content
    state["rate"]=rate
    return state

# 5.Initialising State

In [ ]:
graph=StateGraph(BlogState)

# 6.Adding Node

## 6.1 Adding create blog Node

In [ ]:
graph.add_node("create_blog",create_blog)

## 6.2 Adding Create outline node

In [ ]:
graph.add_node("create_outline",create_outline)

## 6.3 Adding rate the blog node

In [ ]:
graph.add_node("rate_blog",rate_blog)

# 7.Adding Edges

In [ ]:
graph.add_edge(START,"create_outline")
graph.add_edge("create_outline","create_blog")
graph.add_edge("create_blog","rate_blog")
graph.add_edge("rate_blog",END)

# 8.Compilation

In [ ]:
workflow=graph.compile()

In [ ]:
workflow

# 9.Initial State

In [ ]:
initial_state={
    "title":"Rise of AI in the world"
}

# 10.Execution

In [ ]:
result=workflow.invoke(initial_state)
print(result)

In [ ]:
print(result["rate"])